In [1]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
text = "hello월드😁"

print(len(text))
print(list(text))

print(ord('h'))
print(ord('😁'))

print(chr(104))
print(chr(128513))


8
['h', 'e', 'l', 'l', 'o', '월', '드', '😁']
104
128513
h
😁


In [3]:
ids = [ord(char) for char in list(text)]
print(ids)

[104, 101, 108, 108, 111, 50900, 46300, 128513]


In [ ]:
class CharTokenizer:
	def encode(self, text):
		return [ord(char) for char in text]
	def decode(self, ids):
		return ''.join([chr(i) for i in ids])

tokenizer = CharTokenizer()
text = "hello월드😁"

ids = tokenizer.encode(text)
print(ids)

decoded = tokenizer.decode(ids)
print(decoded)

[104, 101, 108, 108, 111, 50900, 46300, 128513]
hello월드😁


In [1]:
encoded = 'A'.encode('utf-8')
print(encoded)
print(list(encoded))

encoded = '가'.encode('utf-8')
print(encoded)
print(list(encoded))

b'A'
[65]
b'\xea\xb0\x80'
[234, 176, 128]


In [3]:
ids = [65]
decoded = bytes(ids).decode('utf-8')
print(decoded)

A


In [ ]:
class ByteTokenizer:
	def encode(self, text):
		return list(text.encode('utf-8'))
	def decode(self, text):
		return bytes(ids).decode('utf-8')

tokenizer = ByteTokenizer()
text = 'hello월드😁'

ids = tokenizer.encode(text)
print(ids)

decoded = tokenizer.decode(ids)
print(decoded)

[104, 101, 108, 108, 111, 236, 155, 148, 235, 147, 156, 240, 159, 152, 129]
hello월드😁


In [ ]:
from collections import defaultdict

def count_pairs(ids):
	counts = defaultdict(int)
	for pair in zip(ids, ids[1:]):
		counts[pair] += 1
	return counts

ids = [1, 2, 3, 1, 2]
counts = count_pairs(ids)
print(counts)

defaultdict(<class 'int'>, {(1, 2): 2, (2, 3): 1, (3, 1): 1})


In [ ]:
def merge(ids, pair, new_id):
	merge_ids = []
	i = 0
	while i < len(ids):
		if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
			merge_ids.append(new_id)
			i += 2
		else:
			merge_ids.append(ids[i])
			i += 1
	return merge_ids

ids = [1, 2, 3, 1, 2]
merged = merge(ids, (1, 2), 4)
print(merged)

[4, 3, 4]


In [ ]:
def train_bpe(text, vocab_size):
	ids = list(text.encode('utf-8'))
	num_merges = vocab_size - 256
	merge_rules = {}

	for step in range(num_merges):
		counts = count_pairs(ids)
		if not counts:
			break
		best_pair = max(counts, key = counts.get)
		new_id = 256 + step
		merge_rules[best_pair] = new_id
		ids = merge(ids, best_pair, new_id)
	return merge_rules

text = 'Hello World! This is BPE training'		# This is의 Th'is is', training의 tra'in''in'g
merge_rules = train_bpe(text, vocab_size = 260)
print(merge_rules)

{(105, 115): 256, (256, 32): 257, (105, 110): 258, (72, 101): 259}


In [ ]:
class BPETokenizer:
	def __init__(self, merge_rules):
		self.merge_rules = merge_rules
		self.id_to_bytes = {i: bytes([i]) for i in range(256)}
		for (id1, id2), new_id in merge_rules.items():
			self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
		self.vocab_size = len(self.id_to_bytes)

	def encode(self, text):
		ids = list(text.encode('utf-8'))
		for merge_pair, new_id in self.merge_rules.items():
			ids = merge(ids, merge_pair, new_id)
		return ids

	def decode(self, ids):
		byte_list = [self.id_to_bytes[i] for i in ids]
		combined_bytes = b''.join(byte_list)
		text = combined_bytes.decode('utf-8', errors = 'replace')
		return text

merge_rules = {(105, 115): 256, (256, 32): 257, (105, 110): 258, (72, 101): 259}

tokenizer = BPETokenizer(merge_rules)

text = 'Hello월드😁'

ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(ids)
print(decoded)


[259, 108, 108, 111, 236, 155, 148, 235, 147, 156, 240, 159, 152, 129]
Hello월드😁


In [9]:
from collections import defaultdict
import re

def count_pairs(ids, counts = None):
	if counts is None:
		counts = defaultdict(int)

	for pair in zip(ids, ids[1:]):
		counts[pair] += 1
	return counts

def merge(ids, pair, new_id):
	merged_ids = []
	i = 0
	while i < len(ids):
		if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
			merged_ids.append(new_id)
			i += 2
		else:
			merged_ids.append(ids[i])
			i += 1
	return merged_ids

def train_bpe(input_text, vocab_size, end_token = '<|endoftext|>'):
	texts = input_text.split(end_token)
	ids_list = [list(text.encode('utf-8')) for text in texts]
	num_merges = vocab_size - 256 - 1
	merge_rules = {}

	for step in range(num_merges):
		counts = defaultdict(int)
		for ids in ids_list:
			counts = count_pairs(ids, counts)
		if not counts:
			break
		best_pair = max(counts, key = counts.get)
		new_id = 256 + step
		merge_rules[best_pair] = new_id
		for i in range(len(ids_list)):
			ids_list[i] = merge(ids_list[i], best_pair, new_id)
	return merge_rules

sample_text = 'Hello world!<|endoftext|>This is BPE training.'
merge_rules = train_bpe(sample_text, vocab_size = 260)

class BPETokenizer:
	def __init__(self, merge_rules, end_token = '<|endoftext|>'):
		self.merge_rules = merge_rules
		self.end_token = end_token
		self.end_token_id = 256 + len(merge_rules)
		self.id_to_bytes = {i: bytes([i]) for i in range(256)}
		for (id1, id2), new_id in merge_rules.items():
			self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
		self.id_to_bytes[self.end_token_id] = self.end_token.encode('utf-8')
		self.vocab_size = len(self.id_to_bytes)

	def _encode_text(self, text):
		ids = list(text.encode('utf-8'))
		for merge_pair, new_id in self.merge_rules.items():
			ids = merge(ids, merge_pair, new_id)
		return ids

	def encode(self, input_text):
		pattern = '(' + re.escape(self.end_token) + ')'
		texts = re.split(pattern, input_text)
		all_ids = []

		for text in texts:
			if text == self.end_token:
				all_ids.append(self.end_token_id)
			else:
				ids = self._encode_text(text)
				all_ids.extend(ids)
		return all_ids

	def decode(self, ids):
		byte_list = [self.id_to_bytes[i] for i in ids]
		text_bytes = b''.join(byte_list)
		text = text_bytes.decode('utf-8', errors = 'replace')
		return text

tokenizer = BPETokenizer(merge_rules)

text = 'Hello world!<|endoftext|>'
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(ids)
print(decoded)

[72, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33, 259]
Hello world!<|endoftext|>


In [10]:
from collections import defaultdict
import regex as re
from tqdm import tqdm

def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)

def count_pairs(ids, counts = None):
	if counts is None:
		counts = defaultdict(int)

	for pair in zip(ids, ids[1:]):
		counts[pair] += 1
	return counts

def merge(ids, pair, new_id):
	merged_ids = []
	i = 0
	while i < len(ids):
		if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
			merged_ids.append(new_id)
			i += 2
		else:
			merged_ids.append(ids[i])
			i += 1
	return merged_ids

def train_bpe(input_text, vocab_size, end_token = '<|endoftext|>'):
	texts = input_text.split(end_token)
	ids_list = []
	for text in texts:
		for pretoken in pretokenize(text):
			ids_list.append(list(pretoken.encode('utf-8')))
	num_merges = vocab_size - 256 - 1
	merge_rules = {}

	for step in tqdm(range(num_merges), desc = 'Training BPE'):
		counts = defaultdict(int)
		for ids in ids_list:
			counts = count_pairs(ids, counts)
		if not counts:
			break
		best_pair = max(counts, key = counts.get)
		new_id = 256 + step
		merge_rules[best_pair] = new_id
		for i in range(len(ids_list)):
			ids_list[i] = merge(ids_list[i], best_pair, new_id)

	return merge_rules

class BPETokenizer:
	def __init__(self, merge_rules, end_token = '<|endoftext|>'):
		self.merge_rules = merge_rules
		self.end_token = end_token
		self.end_token_id = 256 + len(merge_rules)
		self.id_to_bytes = {i: bytes([i]) for i in range(256)}
		for (id1, id2), new_id in merge_rules.items():
			self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
		self.id_to_bytes[self.end_token_id] = self.end_token.encode('utf-8')
		self.vocab_size = len(self.id_to_bytes)

	def _encode_text(self, text):
		ids = list(text.encode('utf-8'))
		for merge_pair, new_id in self.merge_rules.items():
			ids = merge(ids, merge_pair, new_id)
		return ids

	def encode(self, input_text, show_progress = False):
		pattern = '(' + re.escape(self.end_token) + ')'
		texts = re.split(pattern, input_text)
		all_ids = []
		texts = tqdm(texts, desc = 'Encoding') if show_progress else texts
		for text in texts:
			if text == self.end_token:
				all_ids.append(self.end_token_id)
			else:
				for pretoken in pretokenize(text):
					ids = self._encode_text(pretoken)
					all_ids.extend(ids)
		return all_ids

	def decode(self, ids):
		byte_list = [self.id_to_bytes[i] for i in ids]
		text_bytes = b''.join(byte_list)
		text = text_bytes.decode('utf-8', errors = 'replace')
		return text

sample_text = 'Say hello! Why hello? Just hell.<|endoftext|>Good morning!'
merge_rules = train_bpe(sample_text, vocab_size = 270)
tokenizer = BPETokenizer(merge_rules)

text = 'Say hello!'
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(ids)
print(decoded)

for token_id in ids:
	print(f"{token_id} -> '{tokenizer.decode([token_id])}'")

Training BPE: 100%|██████████| 13/13 [00:00<?, ?it/s]

[262, 260, 33]
Say hello!
262 -> 'Say'
260 -> ' hello'
33 -> '!'
